# MSAI6320 — Word Embeddings Lab

**Course:** MSAI6320 — Natural Language Processing  
**Instructor:** Dr. Ahmed I. A. Shawahna | Birzeit University  
**Reference:** Jurafsky & Martin (2026), Chapter 5

---

## 📚 Lab Overview

In this lab, you will explore **Word Embeddings** — a core technique in modern NLP where words are represented as dense numeric vectors in a continuous vector space. Words with similar meanings are positioned close together in this space, enabling arithmetic operations on meaning.

### Learning Objectives

By the end of this lab you will be able to:

1. **Train** your own Word2Vec embeddings on a custom dataset.
2. **Load** pre-trained embeddings from the Gensim model zoo (GloVe).
3. **Explore** word relationships using vector arithmetic (analogies, similarities).
4. **Visualize** high-dimensional word vectors in 2D using PCA and t-SNE.
5. **Build** a simple semantic search engine using phrase embeddings.

---

## 🗂️ Lab Structure

| Section | Description |
|---------|-------------|
| Part 1 | Data loading & tokenization |
| Part 2 | Train Word2Vec from scratch |
| Part 3 | Load pre-trained GloVe embeddings |
| Part 4 | Explore word analogies & similarities |
| Part 5 | Visualize word vectors (PCA & t-SNE) |
| Part 6 | Phrase embeddings & their visualization |
| Part 7 | Semantic search: find similar questions |
| Part 8 | Further tasks (Tasks 7–10) |

---

> **Note:** Run cells **in order** from top to bottom. Each section depends on the ones above it.

**Submitted by** *(pairs; fill in before you submit, see Part 8)*

| | Name | Student ID |
|---|---|---|
| **Student 1** | Rayyan Asia | 1255301 |
| **Student 2** | Ameer Eleyan | 1255304 |

---
## Part 1 — Data Loading & Tokenization

### 📖 The Dataset: Quora Question Pairs

We use a subset of the **Quora Question Pairs** dataset. This dataset contains over **400,000 question pairs** from Quora, a Q&A platform where users ask and answer questions on any topic.

**Why this dataset?**
- Rich, natural English language written by real users.
- Diverse topics: technology, science, politics, culture, etc.
- Short sentences (questions), which is ideal for learning word co-occurrence patterns.

**What we use:**  
We extract individual questions (not pairs) as our training corpus. Each line in `quora.txt` is one question.

### Step 1.1 — Download the Data

In [147]:
# Download the Quora questions dataset from Google Drive
# gdown is a tool that correctly handles Google Drive download links
# (unlike wget, which only downloads the HTML preview page)
# The download is skipped if quora.txt is already here, e.g. when you re-run the notebook.
# If a download was interrupted, delete quora.txt and run this cell again.
import os

if not os.path.exists("./quora.txt"):
    !pip install gdown --quiet
    !gdown "https://drive.google.com/uc?id=173i_GXXPpi0MBvM0GchhwQDzH0djxw_W" -O ./quora.txt
else:
    print("quora.txt already exists: skipping the download.")

# Alternative download link (if the above fails):
# !wget "https://yadi.sk/i/BPQrUu1NaTduEw" -O ./quora.txt

quora.txt already exists: skipping the download.


### Step 1.2 — Load and Inspect the Data

In [148]:
import numpy as np

# Load all lines from the file; each line is one question
data = list(open("./quora.txt", encoding="utf-8"))

print(f"Total number of questions: {len(data):,}")
print(f"\nExample questions (lines 50–52):")
for i, q in enumerate(data[50:53], start=50):
    print(f"  [{i}] {q.strip()}")

Total number of questions: 537,272

Example questions (lines 50–52):
  [50] What TV shows or books help you read people's body language?
  [51] I see this cute girl around my college. I'm too shy to approach her. Is it acceptable to leave her a note with my feelings and number?
  [52] What schools can I apply for with a GRE score of Q-165 V-153?


### Step 1.3 — Tokenization

**What is tokenization?**  
Tokenization splits raw text into individual tokens (words, punctuation marks, etc.). This is the first step in almost every NLP pipeline.

**Why not just use `str.split()`?**  
A simple `split()` on spaces fails to handle:
- Punctuation attached to words: `"word,"` → should be `["word", ","]`
- Contractions: `"don't"` → should be `["don", "'", "t"]`
- Emoticons and special characters

**We use `WordPunctTokenizer` from NLTK**, which splits on punctuation boundaries and handles these cases correctly.

**Task:** Tokenize every question — lowercase all text and split into a list of tokens. The result `data_tok` should be a **list of lists**: one token list per question.

In [149]:
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()

# Demonstrate tokenization on a single example
example = data[50]
print(f"Original : {example.strip()}")
print(f"Tokenized: {tokenizer.tokenize(example.lower())}")

Original : What TV shows or books help you read people's body language?
Tokenized: ['what', 'tv', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [150]:
# TASK: Lowercase each question and tokenize it.
# data_tok is a list of lists — each inner list contains the tokens for one question.

data_tok = []
total_tokens = 0

for sentence in data:
    tokens = tokenizer.tokenize(sentence.lower())  # lowercase + tokenize
    data_tok.append(tokens)
    total_tokens += len(tokens)

print(f"Total questions tokenized : {len(data_tok):,}")
print(f"Total tokens in corpus    : {total_tokens:,}")
print(f"Average tokens per question: {total_tokens / len(data_tok):.1f}")
print(f"\nExample — question 50 tokens:")
print(' '.join(data_tok[50]))

Total questions tokenized : 537,272
Total tokens in corpus    : 7,130,786
Average tokens per question: 13.3

Example — question 50 tokens:
what tv shows or books help you read people ' s body language ?


---
## Part 2 — Training Word2Vec from Scratch

### 📖 What is Word2Vec?

**Word2Vec** (Mikolov et al., 2013) is a neural network-based model that learns dense word representations by predicting words from their context (or vice versa). It comes in two flavors:

- **CBOW (Continuous Bag of Words):** Predicts a target word from its surrounding context words.
- **Skip-gram:** Predicts surrounding context words from a target word.

### Key Hyperparameters

| Parameter | Description | Our Value |
|-----------|-------------|----------|
| `vector_size` | Dimensionality of each word vector | 32 |
| `min_count` | Ignore words appearing fewer than this many times | 5 |
| `window` | Number of context words on each side of the target | 5 |
| `sg` | `sg=0` (the default) trains CBOW, `sg=1` trains Skip-gram | 1 |
| `negative` | Number of noise words per positive pair. 0 turns negative sampling off. | 10 |
| `ns_exponent` | Noise words are drawn from the unigram distribution raised to this power | 0.75 |
| `seed` | Seed for the random initial vectors, so every run gives the same embeddings | 42 |
| `workers` | Number of training threads; one thread is needed for reproducible results | 1 |

> **Note:** We use `vector_size=32` (small) for speed. Production models typically use 100–300 dimensions.

> **Note:** With `seed=42` and `workers=1`, re-running on the same machine gives exactly the same vectors. On a different setup (other versions of NLTK, Gensim or NumPy, or a changed download), the vocabulary or the floating-point arithmetic can differ slightly, so your similarities may differ from your classmates' and from the solutions, typically by a few hundredths. Base your conclusions on clear differences, not on the exact values.

### What is `.wv`?

After training, `model.wv` gives you the **KeyedVectors** object — a lightweight dictionary mapping each word to its learned vector. It supports:
- `model.wv["word"]` or `model.wv.get_vector('word')` → returns the 32-dimensional vector
- `model.wv.most_similar("word")` → finds nearest neighbors
- `model.wv.similarity("word1", "word2")` → cosine similarity

In [151]:
!pip install gensim --quiet

from gensim.models import Word2Vec

print("Training Word2Vec on Quora questions...")

# Train Word2Vec on our tokenized data
# .wv extracts only the word vectors (drops training state to save memory)
model = Word2Vec(
    data_tok,
    vector_size=32,   # each word is a 32-dimensional vector
    min_count=5,      # ignore words that appear fewer than 5 times
    window=5,         # context window: 5 words to the left and right
    sg=1,             # skip-gram (0 = CBOW, the default)
    negative=10,      # 10 negative samples per positive pair
    ns_exponent=0.75, # noise words are drawn from the unigram distribution raised to this power
    seed=42,          # fixed seed for the random initial vectors
    workers=1         # one thread: multi-threaded training is not reproducible, even with a seed
).wv

print(f"\nTraining complete!")
print(f"Vocabulary size : {len(model.key_to_index):,} words")
print(f"Vector dimension: {model.vector_size}")

Training Word2Vec on Quora questions...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'



Training complete!
Vocabulary size : 28,054 words
Vector dimension: 32


In [152]:
# Retrieve the vector for the word 'anything'
vec = model.get_vector('anything')
print(f"Vector for 'anything' (shape={vec.shape}):")
print(vec)

Vector for 'anything' (shape=(32,)):
[-0.50729895 -0.20245476  1.1034267   0.9283417   0.11548291 -0.28966412
 -0.30578122  0.3131561  -0.29150856 -0.23086284 -0.34273255 -0.00156928
 -0.1178669  -0.20652574 -0.1801779  -0.22986852  0.5340958   0.13707423
 -0.14704864 -0.76834244  0.25256282 -0.07925072 -0.748962   -0.9255951
  0.5152682  -0.5800846   0.00629257 -0.1542309  -0.2749562   0.16894892
 -0.290374    0.885284  ]


In [153]:
# Find the 10 most similar words to 'car'
# Returns a list of (word, cosine_similarity) tuples, sorted by similarity
print("Top 10 words most similar to 'car':")
for word, score in model.most_similar('car'):
    print(f"  {word:<15} similarity = {score:.4f}")

Top 10 words most similar to 'car':
  motorcycle      similarity = 0.9509
  truck           similarity = 0.8964
  bicycle         similarity = 0.8848
  vehicle         similarity = 0.8752
  tires           similarity = 0.8584
  tire            similarity = 0.8539
  bike            similarity = 0.8527
  dealership      similarity = 0.8514
  hatchback       similarity = 0.8510
  garage          similarity = 0.8445


### Task 1 — How does the context window change what "similar" means?

`window` decides which words count as the context of a word. Let us see what changing it does to the vectors.

**Task:** Complete `train_sgns(window)` so that it trains Skip-gram on `data_tok` with **exactly the Part 2 settings**, changing only `window`, and returns the word vectors (`.wv`). The test cell trains two models, `window=2` and `window=10`. For *doctor* it prints the similarity to two partners: a **substitute** (a word that can take its place in a sentence) and an **associate** (a word that appears near it in the same question). It also prints the gap *substitute − associate*. Then answer below:

- **(a)** When the window grows from 2 to 10, what happens to the similarity of each partner, and to the gap?
- **(b)** Why? Think about which words fall inside a ±2 window and which only fall inside a ±10 window (the average question is about 13 tokens long, see Step 1.3). Which window would you choose to build a thesaurus, and which to find questions on the same topic?

**Why these words?** The substitute fills the same slot as *doctor* (*ask your doctor / lawyer*), and the associate shares its topic (*the doctor at the hospital*). All three words occur at least 255 times in Quora, so their vectors are well trained, and *lawyer* and *hospital* are almost equally frequent (255 and 257). **Why 2 and 10?** They lie well below and well above Part 2's `window=5`, so the effect is easy to see. Every other setting stays the same, so any difference comes from the window alone.

**Note:** The test trains two models on one thread and takes a few minutes (about 1.5 min for `window=2` and 3.5 min for `window=10` on our machine). Write down your prediction for (a) while it runs. Your numbers may differ slightly from your classmates' (see the note in Part 2), so answer in terms of the direction of each change. The test checks only that the gap shrinks.

**Hint:** Copy the `Word2Vec(...)` call from Part 2.

In [154]:
WINDOW_PAIRS = [("doctor", "lawyer", "hospital")]   # (word, substitute, associate)


def train_sgns(window):
    """Train Skip-gram on data_tok with the Part 2 settings, changing only `window`. Return the word vectors (.wv)."""
    return Word2Vec(
    data_tok,
    vector_size=32,   # each word is a 32-dimensional vector
    min_count=5,      # ignore words that appear fewer than 5 times
    window=window,     # context window
    sg=1,             # skip-gram (0 = CBOW, the default)
    negative=10,      # 10 negative samples per positive pair
    ns_exponent=0.75, # noise words are drawn from the unigram distribution raised to this power
    seed=42,          # fixed seed for the random initial vectors
    workers=1         # one thread: multi-threaded training is not reproducible, even with a seed
    ).wv


In [155]:
# ✅ Test — Task 1
try:
    print("Training Skip-gram with window=2 and window=10 (a few minutes)...")
    task1_models = {w: train_sgns(w) for w in (2, 10)}
except NotImplementedError:
    print("⚠️  Task 1 is not finished yet. Complete train_sgns, re-run its cell, then re-run this test.")
else:
    assert all(m.vector_size == 32 and len(m.key_to_index) == len(model.key_to_index) for m in task1_models.values()), \
        "Keep every Part 2 setting except window (vector_size=32, min_count=5, ...)."
    task1_gaps = {}
    print(f"\n  {'word':<8}{'partner':<20}{'window=2':>10}{'window=10':>11}")
    for word, substitute, associate in WINDOW_PAIRS:
        for label, partner in [("substitute", substitute), ("associate", associate)]:
            sims = [task1_models[w].similarity(word, partner) for w in (2, 10)]
            print(f"  {word:<8}{partner + ' (' + label + ')':<20}{sims[0]:>10.3f}{sims[1]:>11.3f}")
        task1_gaps[word] = [task1_models[w].similarity(word, substitute) - task1_models[w].similarity(word, associate)
                            for w in (2, 10)]
        print(f"  {'':<8}{'gap':<20}{task1_gaps[word][0]:>10.3f}{task1_gaps[word][1]:>11.3f}\n")
    assert all(gap10 < gap2 for gap2, gap10 in task1_gaps.values()), \
        "The gap should shrink from window=2 to window=10. Check that only window differs from Part 2."
    print("✅ Task 1 passed")

Training Skip-gram with window=2 and window=10 (a few minutes)...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl


  word    partner               window=2  window=10
  doctor  lawyer (substitute)      0.869      0.709
  doctor  hospital (associate)     0.766      0.808
          gap                      0.103     -0.100

✅ Task 1 passed


# Answer

a. Going from window 2 to 10, the similarity to lawyer (substitute) drops (0.869 → 0.709) and the similarity to hospital (associate) goes up (0.766 → 0.808), so the gap flips from +0.103 to −0.100.

b. A small window only sees the words right next to a word, which are words that can replace it (doctor / lawyer). A big window sees the whole question, so it also picks up topic words (hospital). For a thesaurus I would use the small window (2), and for finding questions on the same topic the big window (10).

### Task 2 — CBOW or Skip-gram?

Part 2 trained **Skip-gram** (`sg=1`). Gensim's default is **CBOW** (`sg=0`). Let us compare the two on the same data.

**Task:** Complete `train_cbow()` so that it trains **CBOW** on `data_tok` with **exactly the Part 2 settings**, changing only `sg`, and returns the word vectors (`.wv`). The test cell times the training and compares CBOW with the Part 2 Skip-gram `model` on two things: how close the seven **day names** are to each other (the mean cosine similarity over all 21 pairs), and the similarity of one **frequent** pair, *guitar–piano*. Then answer below:

- **(a)** Which model groups the day names better? Why does its training objective help words that occur only a few dozen times?
- **(b)** Why is there no such gap for *guitar–piano*? When would you still choose CBOW?

**Why these words?** Day names form a category nobody disputes, so "they should be close" needs no judgment, and all seven are **rare** in Quora (17–85 occurrences; the test prints the counts). *guitar* and *piano* are a clear pair of **frequent** words (429 and 270 occurrences), so they act as a control: if the two models agree there, a gap on the day names is about word frequency, not about one model being worse overall.

**Note:** Your numbers may differ slightly from your classmates' (see the note in Part 2). Base your answers on clear gaps, not on differences in the second decimal. The test checks only that Skip-gram groups the day names clearly better.

**Hint:** Copy the `Word2Vec(...)` call from Part 2. For (b), compare the training time with the Part 2 training cell (about 2.5–3 minutes on our machine).

In [156]:
DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]   # rare in Quora
FREQUENT_PAIR = ("guitar", "piano")                                                      # frequent in Quora


def train_cbow():
    return Word2Vec(
    data_tok,
    vector_size=32,   # each word is a 32-dimensional vector
    min_count=5,      # ignore words that appear fewer than 5 times
    sg=0,             # CBOW (1 = skip-gram, the default)
    negative=10,      # 10 negative samples per positive pair
    ns_exponent=0.75, # noise words are drawn from the unigram distribution raised to this power
    seed=42,          # fixed seed for the random initial vectors
    workers=1         # one thread: multi-threaded training is not reproducible, even with a seed
    ).wv


In [157]:
# ✅ Test — Task 2
import time
from itertools import combinations


def mean_pairwise_similarity(wv, words):
    """Mean cosine similarity over all pairs of `words`."""
    return float(np.mean([wv.similarity(a, b) for a, b in combinations(words, 2)]))


try:
    print("Training CBOW...")
    start = time.time()
    task2_cbow = train_cbow()
    task2_seconds = time.time() - start
except NotImplementedError:
    print("⚠️  Task 2 is not finished yet. Complete train_cbow, re-run its cell, then re-run this test.")
else:
    assert task2_cbow.vector_size == 32 and len(task2_cbow.key_to_index) == len(model.key_to_index), \
        "Keep every Part 2 setting except sg (vector_size=32, min_count=5, ...)."
    print(f"CBOW training time: {task2_seconds:.0f} s\n")
    print("Occurrences in Quora:", {w: int(model.get_vecattr(w, "count")) for w in DAYS + list(FREQUENT_PAIR)}, "\n")
    task2_days = {"Skip-gram": mean_pairwise_similarity(model, DAYS), "CBOW": mean_pairwise_similarity(task2_cbow, DAYS)}
    task2_pair = {"Skip-gram": model.similarity(*FREQUENT_PAIR), "CBOW": task2_cbow.similarity(*FREQUENT_PAIR)}
    print(f"  {'':<30}{'Skip-gram':>10}{'CBOW':>8}")
    print(f"  {'day names (mean of 21 pairs)':<30}{task2_days['Skip-gram']:>10.3f}{task2_days['CBOW']:>8.3f}")
    print(f"  {'guitar–piano':<30}{task2_pair['Skip-gram']:>10.3f}{task2_pair['CBOW']:>8.3f}")
    assert task2_days["Skip-gram"] - task2_days["CBOW"] > 0.1, \
        "Skip-gram should group the day names clearly better than CBOW. Check that only sg differs from Part 2."
    print("✅ Task 2 passed")

Training CBOW...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


CBOW training time: 32 s

Occurrences in Quora: {'monday': 50, 'tuesday': 17, 'wednesday': 18, 'thursday': 19, 'friday': 85, 'saturday': 43, 'sunday': 61, 'guitar': 429, 'piano': 270} 

                                 Skip-gram    CBOW
  day names (mean of 21 pairs)       0.801   0.612
  guitar–piano                       0.910   0.940
✅ Task 2 passed


# Answer

a. Skip-gram. Its day names are closer together (0.801 vs 0.612 for CBOW). Skip-gram treats every neighbour as its own training example, so even a rare word gets many updates. CBOW mixes the neighbours into one average, so a rare word only gets a small share.

b. Guitar and piano are common, so both models have enough examples and score about the same (0.910 vs 0.940). I would still choose CBOW when I want faster training (it took 32 s) or when the words are mostly common ones.

---
## Part 3 — Loading Pre-trained GloVe Embeddings

### 📖 What is GloVe?

**GloVe (Global Vectors for Word Representation)** (Pennington et al., 2014) is an alternative to Word2Vec. Instead of using a sliding window, GloVe builds a **global co-occurrence matrix** across the entire corpus and factorizes it.

### Why use pre-trained embeddings?

Training on a small corpus (like our ~537k questions) produces limited embeddings. Pre-trained embeddings are trained on **billions of words** and capture much richer semantic relationships.

### Available pre-trained models via Gensim

| Model Name | Training Data | Vocabulary | Dimensions |
|------------|--------------|------------|------------|
| `glove-twitter-25` | 2B tweets, 27B tokens | 1.2M words | 25 |
| `glove-twitter-100` | 2B tweets, 27B tokens | 1.2M words | 100 |
| `glove-wiki-gigaword-50` | Wikipedia 2014 + Gigaword 5 (6B tokens) | 400K words | 50 |
| **`glove-wiki-gigaword-100`** | Wikipedia 2014 + Gigaword 5 (6B tokens) | 400K words | **100** |
| `glove-wiki-gigaword-300` | Wikipedia 2014 + Gigaword 5 (6B tokens) | 400K words | 300 |
| `word2vec-google-news-300` | Google News (100B tokens) | 3M words | 300 |

**We use `glove-wiki-gigaword-100`:**
- Trained on **Wikipedia 2014** + **Gigaword 5** (English newswire)
- **6 billion tokens**, **400,000 word vocabulary**
- Each word is a **100-dimensional** vector
- Well-balanced between formal (Wikipedia) and journalistic (Gigaword) language

> **First download may take a few minutes** depending on your internet speed.

In [158]:
import gensim.downloader as api

print("Loading pre-trained GloVe embeddings (glove-wiki-gigaword-100)...")
print("Training data: Wikipedia 2014 + Gigaword 5 | 6B tokens | 400K vocab | 100 dims")
print("This may take a few minutes on the first run (downloading ~128 MB)...\n")

# Load the pre-trained GloVe model
model = api.load('glove-wiki-gigaword-100')

print(f"\nModel loaded successfully!")
print(f"Type          : {type(model)}")
print(f"Vocabulary size: {len(model.key_to_index):,} words")
print(f"Vector dimension: {model.vector_size}")

Loading pre-trained GloVe embeddings (glove-wiki-gigaword-100)...
Training data: Wikipedia 2014 + Gigaword 5 | 6B tokens | 400K vocab | 100 dims
This may take a few minutes on the first run (downloading ~128 MB)...


Model loaded successfully!
Type          : <class 'gensim.models.keyedvectors.KeyedVectors'>
Vocabulary size: 400,000 words
Vector dimension: 100


In [159]:
# Get the 100-dimensional vector for the word 'bread'
vec = model.get_vector('bread')
print(f"Vector for 'bread' (shape={vec.shape}):")
print(vec)

Vector for 'bread' (shape=(100,)):
[-0.66146    0.94335   -0.72214    0.17403   -0.42524    0.36303
  1.0135    -0.14802    0.25817   -0.20326   -0.64338    0.16632
  0.61518    1.397     -0.094506   0.0041843 -0.18976   -0.55421
 -0.39371   -0.22501   -0.34643    0.32076    0.34395   -0.7034
  0.23932    0.69951   -0.16461   -0.31819   -0.34034   -0.44906
 -0.069667   0.35348    0.17498   -0.95057   -0.2209     1.0647
  0.23231    0.32569    0.47662   -1.1206     0.28168   -0.75172
 -0.54654   -0.66337    0.34804   -0.69058   -0.77092   -0.40167
 -0.069351  -0.049238  -0.39351    0.16735   -0.14512    1.0083
 -1.0608    -0.87314   -0.29339    0.68278    0.61634   -0.088844
  0.88094    0.099809  -0.27161   -0.58026    0.50364   -0.93814
  0.67576   -0.43124   -0.10517   -1.2404    -0.74353    0.28637
  0.29012    0.89377    0.67406    0.86422   -0.30693   -0.14718
  0.078353   0.74013    0.32658   -0.052579  -1.1665     0.87079
 -0.69402   -0.75977   -0.37164   -0.11887    0.18551    

In [160]:
# Find the 10 most similar words to 'usa'
print("Top 10 words most similar to 'usa':")
for word, score in model.most_similar('usa'):
    print(f"  {word:<15} similarity = {score:.4f}")

Top 10 words most similar to 'usa':
  canada          similarity = 0.6544
  america         similarity = 0.6452
  u.s.a.          similarity = 0.6184
  united          similarity = 0.6017
  states          similarity = 0.5971
  australia       similarity = 0.5839
  world           similarity = 0.5590
  2010            similarity = 0.5581
  2012            similarity = 0.5504
  davis           similarity = 0.5464


---
## Part 4 — Word Similarities & Analogies

### 📖 Cosine Similarity

The similarity between two word vectors is measured using **cosine similarity**:

$$\text{similarity}(\vec{u}, \vec{v}) = \cos(\theta) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \cdot \|\vec{v}\|}$$

- **Score = 1.0** → identical direction (very similar meaning)
- **Score = 0.0** → perpendicular (unrelated)
- **Score = −1.0** → opposite direction

### 📖 Word Analogies with Vector Arithmetic

One of the most famous properties of word embeddings is that **semantic relationships are encoded as vector directions**:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

This means: "man is to king as woman is to queen"

The `most_similar(positive=[...], negative=[...])` method implements this:
- `positive` words: **add** their vectors
- `negative` words: **subtract** their vectors

---


### 4.1 — A Note on `negative` in `most_similar`

⚠️ **Common Misconception:** The `negative` parameter in `most_similar` does **not** find antonyms.

```python
model.most_similar(negative='white')   # NOT antonyms of 'white'
```

It performs vector arithmetic: finds words closest to the **negative** direction of the `white` vector (i.e., `-white`). Antonyms are **not** guaranteed to be opposite in vector space — in fact, antonyms often appear in **similar contexts** and end up as *neighbors*, not opposites:

- "The color is white"  
- "The color is black"

Both `white` and `black` occur after "The color is", so they are **close** in vector space.

In [161]:
# This finds words in the OPPOSITE direction of 'white' — NOT antonyms
print("Words in the opposite direction of 'white' (NOT antonyms):")
for word, score in model.most_similar(negative=['white']):
    print(f"  {word:<15} score = {score:.4f}")

Words in the opposite direction of 'white' (NOT antonyms):
  basadre         score = 0.6282
  oudomxay        score = 0.6226
  ___________________________________________________________ score = 0.6052
  oodle           score = 0.6013
  riep            score = 0.5983
  tingstrom       score = 0.5918
  epigenesis      score = 0.5897
  stojaković      score = 0.5892
  marinne         score = 0.5883
  phrabang        score = 0.5880


### 4.2 — Basic Similarity

In [162]:
# Cosine similarity between two words (built into Gensim)
pairs = [("white", "black"), ("king", "queen"), ("dog", "cat"), ("happy", "sad")]

print("Cosine similarity between word pairs:")
print(f"  {'Word 1':<10} {'Word 2':<10} {'Similarity':>12}")
print("  " + "-" * 35)
for w1, w2 in pairs:
    sim = model.similarity(w1, w2)
    print(f"  {w1:<10} {w2:<10} {sim:>12.4f}")

Cosine similarity between word pairs:
  Word 1     Word 2       Similarity
  -----------------------------------
  white      black            0.8652
  king       queen            0.7508
  dog        cat              0.8798
  happy      sad              0.6801


### Task 3 — Is "similar" the same as "synonym"?

In §4.1 we saw that *white* and *black* end up close because they occur in the same contexts. Let us see how far this goes.

**Task:** Complete `antonym_wins(word, antonym, synonym)` so that it returns `True` when `word` is more similar (cosine) to its antonym than to its synonym. The test cell runs it on the five triples in `TRIPLES` and prints the results. Then answer below:

- **(a)** In how many triples does the antonym win, and why can an antonym be closer than a synonym?
- **(b)** What would this mean if you used `most_similar` to build a thesaurus or to expand search queries?

**Why these triples?** All 15 words are common single words in GloVe's vocabulary, so none of the vectors is poorly trained (§5 explains why rare words are noisy), and each synonym can replace its word in most sentences.

**Hint:** `model.similarity(w1, w2)` from §4.2.

In [163]:
TRIPLES = [("good", "bad", "nice"), ("hot", "cold", "warm"), ("increase", "decrease", "rise"),
           ("big", "small", "large"), ("happy", "sad", "glad")]     # (word, antonym, synonym)


def antonym_wins(word, antonym, synonym):
    return model.similarity(word, antonym) > model.similarity(word, synonym)


In [164]:
# ✅ Test — Task 3
try:
    task3_wins = [antonym_wins(*triple) for triple in TRIPLES]
except NotImplementedError:
    print("⚠️  Task 3 is not finished yet. Complete antonym_wins, re-run its cell, then re-run this test.")
else:
    assert all(isinstance(w, (bool, np.bool_)) for w in task3_wins), "antonym_wins must return True or False."
    for triple, win in zip(TRIPLES, task3_wins):
        print(f"  {triple[0]:<9} antonym wins: {win}")
    assert sum(task3_wins) == 4, (f"The antonym should win in 4 of the 5 triples, but you report {sum(task3_wins)}. "
                                  "Compare model.similarity(word, antonym) with model.similarity(word, synonym).")
    print("✅ Task 3 passed")

  good      antonym wins: True
  hot       antonym wins: True
  increase  antonym wins: True
  big       antonym wins: True
  happy     antonym wins: False
✅ Task 3 passed


# Answer
a.

 The antonym wins in 4 of the 5 triples (all except happy). Similarity here does not necessarily mean antonym of the word, it could mean the topic is similar which is why the difference between the embeddings, as they are the two sides of the same topic

b.

You would pull antonyms instead of synonyms, which would be a poor thesaurus, and query expansion would add words with the opposite meaning.



### 4.3 — Word Analogies

The analogy function implements: **x1 : x2 :: y1 : ?**

In vector arithmetic: `result ≈ x2 + y1 - x1`

| Analogy | Formula |
|---------|--------|
| `man : king :: woman : ?` | `king + woman - man` → `queen` |
| `france : paris :: italy : ?` | `paris + italy - france` → `rome` |

In [165]:
def analogy(x1, x2, y1):
    """
    Solve the word analogy: x1 is to x2 as y1 is to ?

    Vector arithmetic: result ≈ x2 + y1 - x1

    Parameters:
        x1 (str): Source word (to subtract)
        x2 (str): Target word (to add)
        y1 (str): Query word (to add)

    Returns:
        str: The word that best completes the analogy
    """
    result = model.most_similar(positive=[x2, y1], negative=[x1])
    return result[0][0]

In [166]:
answer = analogy("man", "king", "woman")
print(f"man:king :: woman:? → {answer}")

man:king :: woman:? → queen


In [167]:
answer = analogy("france", "paris", "italy")
print(f"france:paris :: italy:? → {answer}")

france:paris :: italy:? → rome


In [168]:
answer = analogy("pencil", "sketching", "camera")
print(f"pencil:sketching :: camera:? → {answer}")

pencil:sketching :: camera:? → photographing


In [169]:
answer = analogy("tall", "tallest", "long")
print(f"tall:tallest :: long:? → {answer}")

tall:tallest :: long:? → longest


In [170]:
answer = analogy("peace", "white", "war")
print(f"peace:white :: war:? → {answer}")

peace:white :: war:? → black


In [171]:
answer = analogy("teacher", "school", "professor")
print(f"teacher:school :: professor:? → {answer}")

teacher:school :: professor:? → university


In [172]:
answer = analogy("christian", "church", "muslim")
print(f"christian:church :: muslim:? → {answer}")

christian:church :: muslim:? → mosque


### Task 4 — What does `analogy()` hide?

`analogy()` returns the top result of `most_similar`, and `most_similar` never returns any of the words you gave it.

**Task:** Complete `analogy_scores(x1, x2, y1, candidates)`: build $\vec{x_2}+\vec{y_1}-\vec{x_1}$ with `model.get_vector`, and return a dict mapping each candidate to its cosine similarity with that vector, using the formula from Part 4 (NumPy only). The test runs `man:king :: woman:?` with candidates `king, queen`, and `france:paris :: italy:?` with candidates `paris, italy, rome`. Then answer below:

- **(a)** If the input words were allowed as answers, what would each analogy return?
- **(b)** Why does this happen for one analogy and not the other?
- **(c)** What does this say about headline claims like "king − man + woman = queen"?

**Why compute it ourselves?** `most_similar` filters out the input words, so it cannot show us this. **Why these two?** They are the worked examples in §4.3, and both look like successes there.

In [173]:
def analogy_scores(x1, x2, y1, candidates):
    """Score candidate answers to x1 : x2 :: y1 : ? without excluding any word.

    Builds v = x2 + y1 - x1 from the raw vectors (model.get_vector) and returns
    a dict {candidate: cosine similarity between v and the candidate's vector}.
    """
    v = model.get_vector(x2) + model.get_vector(y1) - model.get_vector(x1)
    return {c: float(np.dot(v, model.get_vector(c)) / (np.linalg.norm(v) * np.linalg.norm(model.get_vector(c))))
            for c in candidates}

In [174]:
# ✅ Test — Task 4
try:
    task4_royal = analogy_scores("man", "king", "woman", ["king", "queen"])
    task4_capital = analogy_scores("france", "paris", "italy", ["paris", "italy", "rome"])
except NotImplementedError:
    print("⚠️  Task 4 is not finished yet. Complete analogy_scores, re-run its cell, then re-run this test.")
else:
    for name, scores in [("man:king :: woman:?", task4_royal), ("france:paris :: italy:?", task4_capital)]:
        ranked = sorted(scores.items(), key=lambda item: -item[1])
        print(f"{name:<25}" + "   ".join(f"{w} {s:.4f}" for w, s in ranked))
    assert set(task4_royal) == {"king", "queen"} and set(task4_capital) == {"paris", "italy", "rome"}, \
        "Return a dict with one entry per candidate word."
    assert np.isclose(task4_royal["queen"], 0.7834, atol=1e-3), (f"cos(v, queen) should be about 0.783, got {task4_royal['queen']:.4f}. "
                                                         "Use v = king + woman - man from model.get_vector, and divide by both norms.")
    assert np.isclose(task4_royal["king"], 0.8552, atol=1e-3), f"cos(v, king) should be about 0.855, got {task4_royal['king']:.4f}."
    assert np.isclose(task4_capital["rome"], 0.8084, atol=1e-3), f"cos(v, rome) should be about 0.808, got {task4_capital['rome']:.4f}."
    assert np.isclose(task4_capital["paris"], 0.6912, atol=1e-3), f"cos(v, paris) should be about 0.691, got {task4_capital['paris']:.4f}."
    print("✅ Task 4 passed")

man:king :: woman:?      king 0.8552   queen 0.7834
france:paris :: italy:?  rome 0.8084   italy 0.7273   paris 0.6912
✅ Task 4 passed


# Answer

a. `man:king :: woman:?` would return **king** (0.855), because it is closer than queen (0.783). `france:paris :: italy:?` would return **rome** (0.808), which beats paris (0.691).

b. man, king, woman and queen are used near each other, so the vector stays close to king. france and paris are not that close, so the vector moves away from paris and lands near rome.

c. "king − man + woman = queen" only works because `most_similar` hides the input words. Without that, the answer would be king.

---
## Part 5 — Visualizing Word Vectors

### 📖 The Dimensionality Reduction Challenge

Our GloVe vectors live in **100-dimensional** space — impossible to visualize directly. We use **dimensionality reduction** to project them into 2D while preserving as much structure as possible.

We will compare two methods:

| Method | Type | Best for | Speed |
|--------|------|----------|-------|
| **PCA** | Linear | Global structure, overview | Fast |
| **t-SNE** | Non-linear | Local clusters, neighborhoods | Slow |

### 📖 Why the top 1,000 most frequent words?

- **Frequent words** appear more often in training data → their vectors are more reliably trained.
- **Rare words** appear few times → their vectors are noisy and less meaningful.
- 1,000 words is a good balance: enough to see structure, not too many to clutter the plot.

### Step 5.1 — Select the Top 1,000 Words

In [175]:
# Get all vocabulary words, sort by frequency (descending), keep top 1000
#
# model.key_to_index   → dictionary: {word: index}
# model.get_vecattr(word, 'count') → GloVe ships no real counts, so gensim stores placeholder
#     counts in rank order (vocab_size - rank). The GloVe file lists words from most to least
#     frequent, so sorting by these counts still puts the most frequent words first.
# sorted(..., reverse=True)[:1000] → keep only top 1000 most frequent

words = sorted(
    model.key_to_index.keys(),
    key=lambda word: model.get_vecattr(word, 'count'),
    reverse=True
)[:1000]

print(f"Number of words selected: {len(words)}")
print(f"Top 20 most frequent words: {words[:20]}")

Number of words selected: 1000
Top 20 most frequent words: ['the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s", 'for', '-', 'that', 'on', 'is', 'was', 'said', 'with', 'he', 'as']


In [176]:
# Get the embedding vector for each of the top 1000 words
word_vectors = [model.get_vector(word) for word in words]

print(f"Word vectors shape: {len(word_vectors)} words × {len(word_vectors[0])} dimensions")

Word vectors shape: 1000 words × 100 dimensions


### Step 5.2 — PCA: Linear Dimensionality Reduction

**Principal Component Analysis (PCA)** finds the axes of maximum variance in the data and projects onto them.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">

**Mathematical formulation:**

PCA decomposes the data matrix $X \in \mathbb{R}^{n \times m}$ into two smaller matrices $W$ and $\hat{W}$, minimizing reconstruction error:

$$\|(X W) \hat{W} - X\|^2_2 \to \min_{W, \hat{W}}$$

- $X \in \mathbb{R}^{n \times m}$ — input data matrix (centered; $n$ words, $m=100$ dimensions)
- $W \in \mathbb{R}^{m \times d}$ — projection matrix (maps 100D → 2D)
- $\hat{W} \in \mathbb{R}^{d \times m}$ — reconstruction matrix (maps 2D → 100D)
- $d=2$ — target dimensions for visualization

**After PCA**, we standardize (zero mean, unit variance) so the plot is well-scaled.

In [177]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Step 1: Reduce 100D vectors → 2D using PCA
pca = PCA(n_components=2, random_state=42)
word_vectors_pca = pca.fit_transform(word_vectors)

print(f"Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print(f"Total variance captured  : {sum(pca.explained_variance_ratio_):.1%}")
print(f"\nPCA output shape: {word_vectors_pca.shape}")
print(f"First 5 rows (2D coordinates):")
print(word_vectors_pca[:5])

Explained variance ratio: PC1=7.0%, PC2=6.1%
Total variance captured  : 13.1%

PCA output shape: (1000, 2)
First 5 rows (2D coordinates):
[[-0.0151126  -0.01992977]
 [ 0.12769052  0.20456001]
 [-0.24557317  0.62642907]
 [ 0.07698303 -0.22491154]
 [-0.88268129 -0.17791064]]


In [178]:
# Step 2: Standardize the PCA output (zero mean, unit variance)
scaler = StandardScaler()
word_vectors_pca = scaler.fit_transform(word_vectors_pca)

print(f"After standardization:")
print(f"  Mean  : {word_vectors_pca.mean(axis=0).round(4)}  (should be ≈ 0)")
print(f"  Std   : {word_vectors_pca.std(axis=0).round(4)}   (should be ≈ 1)")

After standardization:
  Mean  : [0. 0.]  (should be ≈ 0)
  Std   : [1. 1.]   (should be ≈ 1)


### Step 5.3 — Interactive Visualization with Bokeh

We use **Bokeh** to create an **interactive** scatter plot:
- **Each dot** represents one word
- **Position** reflects semantic similarity (nearby dots ≈ similar meaning)
- **Hover** your mouse over any dot to see which word it represents
- **Scroll** to zoom in/out; **click and drag** to pan

In [179]:
import bokeh.models as bm
import bokeh.plotting as pl
from bokeh.io import output_notebook

output_notebook()  # render Bokeh plots inline in the notebook


def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=700, height=500, show=True, **kwargs):
    """
    Draw an interactive 2D scatter plot of word vectors.

    Parameters:
        x, y    : 1D arrays of x and y coordinates
        radius  : dot size
        alpha   : dot transparency (0=invisible, 1=solid)
        color   : dot color (string or list of strings)
        width, height: plot dimensions in pixels
        **kwargs: extra columns for the hover tooltip (e.g., token=words)

    Interaction:
        - Hover over a dot to see its label
        - Scroll to zoom
        - Drag to pan
    """
    if isinstance(color, str):
        color = [color] * len(x)

    data_source = bm.ColumnDataSource({'x': x, 'y': y, 'color': color, **kwargs})

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height,
                    title="Word Vector Visualization")
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)
    fig.add_tools(bm.HoverTool(tooltips=[(key, '@' + key) for key in kwargs.keys()]))

    if show:
        pl.show(fig)
    return fig

print("draw_vectors() function defined. Run the next cell to plot.")

Loading BokehJS ...

draw_vectors() function defined. Run the next cell to plot.


In [180]:
# Plot the 1000 most frequent words in 2D PCA space
# Hover over any dot to see the word it represents
draw_vectors(
    word_vectors_pca[:, 0],   # x-axis = first principal component
    word_vectors_pca[:, 1],   # y-axis = second principal component
    token=words               # tooltip: show the word when hovering
)

# Try to identify clusters — can you spot groups of numbers, countries, verbs?

figure(id='p1266', ...)

### Task 5 — How much of the 100-D space does the PCA plot show?

**Task:** Complete `components_needed(vectors, share)` to return the smallest number of principal components whose explained variance adds up to at least `share`, using the same 1,000 `word_vectors`. The test prints the result for 50% and 80%. Then answer below:

- **(a)** What share of the variance does the 2-D plot above show, and how many components would you need for half?
- **(b)** Two words sit right next to each other in the plot. Can you conclude that they are similar? How would you check?

**Why fit `PCA()` with all components?** To add up the first k ratios we need all of them, and with all 100 components scikit-learn uses the exact SVD, so the answer is deterministic. **Why 50% and 80%?** There is no standard cut-off: 50% means "half the spread", and 80% is a common rule of thumb.

**Hint:** `np.cumsum(PCA().fit(vectors).explained_variance_ratio_)`. Remember indices start at 0.

In [181]:
def components_needed(vectors, share):
    """Return the smallest number of principal components whose explained variance adds up to at least `share`.

    vectors: [n × d] word vectors (here 1000 × 100)    share: a fraction in (0, 1], e.g. 0.5 for 50%
    """
    # YOUR CODE HERE
    cumsum = np.cumsum(PCA().fit(np.asarray(vectors)).explained_variance_ratio_)
    k = np.argmax(cumsum >= share) + 1
    return k

In [182]:
# ✅ Test — Task 5
try:
    task5_k50 = components_needed(word_vectors, 0.5)
    task5_k80 = components_needed(word_vectors, 0.8)
except NotImplementedError:
    print("⚠️  Task 5 is not finished yet. Complete components_needed, re-run its cell, then re-run this test.")
else:
    print(f"Components needed for 50% of the variance: {task5_k50}")
    print(f"Components needed for 80% of the variance: {task5_k80}")
    assert task5_k50 != 15, "You returned 15 for 50%: an off-by-one error. The index of the first component that reaches the share is one less than the count."
    assert task5_k50 == 16, f"16 components are needed for 50%, but you returned {task5_k50}. Fit PCA() on word_vectors with all components."
    assert task5_k80 == 47, f"47 components are needed for 80%, but you returned {task5_k80}."
    print("✅ Task 5 passed")

Components needed for 50% of the variance: 16
Components needed for 80% of the variance: 47
✅ Task 5 passed


# Answer

a. The 2-D plot only shows about 13% of the variance (PC1 = 7.0%, PC2 = 6.1%). About 16 components would be needed for half (47 for 80%).

b. No. Close in the plot does not mean similar, because about 87% of the information is thrown away. To check, compare their cosine similarity in the full 100-D space (`model.similarity`).

### Step 5.4 — t-SNE: Non-linear Dimensionality Reduction

**t-SNE (t-Distributed Stochastic Neighbor Embedding)** is much better at revealing **local cluster structure** than PCA:

| | PCA | t-SNE |
|--|-----|-------|
| Type | Linear | Non-linear |
| Preserves | Global structure | Local neighborhoods |
| Speed | Very fast | Slow (minutes for 1000 points) |
| Deterministic | ✅ Yes | ❌ No (random seed matters) |
| Cluster quality | Medium | Excellent |

**`verbose=2`** prints a progress update every 50 iterations so you can see it's still running.

> ⏳ **This cell takes 1–3 minutes.** This is normal.

In [183]:
from sklearn.manifold import TSNE

print("Running t-SNE on 1000 word vectors (100D → 2D)...")
print("This may take 1–3 minutes. Progress is printed every 50 iterations.\n")

# t-SNE: reduce 100D word vectors to 2D
# verbose=2 → print progress every 50 iterations
# init and learning_rate are set explicitly: newer scikit-learn versions changed their defaults
tsne = TSNE(n_components=2, init='random', learning_rate=200.0, verbose=2, random_state=42)
word_vectors_tsne = tsne.fit_transform(np.array(word_vectors))

# Standardize the output (zero mean, unit variance)
scaler_tsne = StandardScaler()
word_vectors_tsne = scaler_tsne.fit_transform(word_vectors_tsne)

print(f"\nt-SNE complete! Output shape: {word_vectors_tsne.shape}")

Running t-SNE on 1000 word vectors (100D → 2D)...
This may take 1–3 minutes. Progress is printed every 50 iterations.

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 1000 samples in 0.001s...
[t-SNE] Computed neighbors for 1000 samples in 0.050s...


[t-SNE] Computed conditional probabilities for sample 1000 / 1000
[t-SNE] Mean sigma: 1.560991
[t-SNE] Computed conditional probabilities in 0.019s
[t-SNE] Iteration 50: error = 78.1045990, gradient norm = 0.2228794 (50 iterations in 0.133s)
[t-SNE] Iteration 100: error = 79.9375610, gradient norm = 0.2251764 (50 iterations in 0.155s)
[t-SNE] Iteration 150: error = 79.3544159, gradient norm = 0.1856008 (50 iterations in 0.153s)
[t-SNE] Iteration 200: error = 79.2894745, gradient norm = 0.1802519 (50 iterations in 0.129s)
[t-SNE] Iteration 250: error = 79.4931183, gradient norm = 0.2190138 (50 iterations in 0.160s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 79.493118
[t-SNE] Iteration 300: error = 1.8308121, gradient norm = 0.0050407 (50 iterations in 0.136s)
[t-SNE] Iteration 350: error = 1.6939676, gradient norm = 0.0043133 (50 iterations in 0.117s)
[t-SNE] Iteration 400: error = 1.6380966, gradient norm = 0.0028728 (50 iterations in 0.142s)
[t-SNE] Iteration 

In [184]:
# Plot words in t-SNE space (green dots)
draw_vectors(
    word_vectors_tsne[:, 0],
    word_vectors_tsne[:, 1],
    color='green',
    token=words
)

# t-SNE should reveal tighter clusters than PCA
# Look for: number clusters, country clusters, verb clusters, etc.

figure(id='p1317', ...)

---
## Part 6 — Phrase Embeddings & Semantic Search

### 📖 From Words to Phrases

Individual words are great, but real queries are **phrases or sentences**. We can represent a phrase as a single vector by **averaging** the vectors of its constituent words:

$$\vec{\text{phrase}} = \frac{1}{|V|} \sum_{w \in V} \vec{w}$$

where $V$ is the set of words in the phrase that exist in the model vocabulary.

**Why averaging?**  
It's simple, fast, and works surprisingly well for short texts. More sophisticated methods (e.g., TF-IDF weighted averaging, sentence transformers) exist but require more complexity.

### 📖 Semantic Search

Once we have phrase vectors, we can find **semantically similar questions** by measuring cosine similarity between query and candidate phrase vectors. This is the core of many search and recommendation systems.

---

### Step 6.1 — The `get_phrase_embedding` Function

**Important fix:** Before tokenizing, we remove all non-alphabetic characters using `re.sub`. This ensures that punctuation tokens like `'`, `.`, `...` don't get their own (potentially noisy) vectors mixed into the phrase average.

In [185]:
import re
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()


def get_phrase_embedding(phrase):
    """
    Convert a phrase (string) into a single vector by averaging word embeddings.

    Pipeline:
        1. Remove non-alphabetic characters (keeps only letters and spaces)
        2. Lowercase the text
        3. Tokenize into words
        4. Look up each word's embedding vector
        5. Average the vectors (skip words not in vocabulary)
        6. If NO words are in vocabulary, return a zero vector

    Parameters:
        phrase (str): Input text (question, sentence, etc.)

    Returns:
        np.ndarray: A vector of shape (model.vector_size,)
    """
    # Start with a zero vector of the same dimension as the model
    vector = np.zeros([model.vector_size], dtype='float32')

    # Remove non-alphabetic characters (keep only letters and spaces)
    phrase = re.sub(r"[^a-zA-Z ]", " ", phrase)

    # Lowercase and tokenize
    tokens = tokenizer.tokenize(phrase.lower())

    # Accumulate vectors for words that exist in the vocabulary
    word_count = 0
    for word in tokens:
        if word in model:
            vector += model.get_vector(word)
            word_count += 1

    # Compute the average (if at least one word was found)
    if word_count > 0:
        vector /= word_count
    # If word_count == 0, we return the zero vector as-is

    return vector


print("get_phrase_embedding() function defined.")

get_phrase_embedding() function defined.


In [186]:
# Test the function
test_phrase = "I'm very sure. This never happened to me before..."
vec = get_phrase_embedding(test_phrase)

print(f"Input phrase : '{test_phrase}'")
print(f"Output vector shape: {vec.shape}")
print(f"Values    : {vec}")

Input phrase : 'I'm very sure. This never happened to me before...'
Output vector shape: (100,)
Values    : [-0.065415    0.27285913  0.41931802 -0.1741391  -0.543794    0.16832659
  0.0181858   0.3089701   0.06807189 -0.116418    0.57428527  0.18987161
  0.1855854  -0.03273772  0.01693181 -0.31264704 -0.1282818   0.3549458
 -0.6717396   0.313627    0.28782818  0.135679   -0.0463049  -0.13910802
  0.22647336  0.20120879 -0.1683826  -0.63071805  0.43988925 -0.21592398
 -0.21157572  0.61850995  0.25661188  0.1383686   0.01048398  0.3025121
 -0.1947878   0.26004422  0.11440068 -0.2468426  -0.34485298 -0.17943957
  0.18634799 -0.55397666 -0.3266223  -0.0520781   0.3725071  -0.3515661
 -0.0087364  -1.0674479   0.1419605  -0.068459    0.372315    1.045062
 -0.2981836  -2.52842    -0.06764711  0.1238629   1.4936041   0.4000618
 -0.18165721  1.0030043  -0.5172162  -0.05511557  0.889149    0.0779058
  0.59133106  0.4755695   0.01873438 -0.1349611   0.073339   -0.30428702
  0.06305479 -0.4447059

### Step 6.2 — Visualize Phrase Embeddings

We sample ~1,000 questions evenly from the full dataset, compute their phrase vectors, and visualize them with t-SNE.

**`data[::step]`** — Python slice notation with a step size:  
- `len(data) // 1000` gives the step (every Nth element)
- This ensures we sample **evenly** across the entire dataset, not just the beginning

In [187]:
# Sample ~1000 questions evenly from the full dataset
step = len(data) // 1000
chosen_phrases = data[::step]

print(f"Total questions in dataset : {len(data):,}")
print(f"Sampling every {step}th question  → {len(chosen_phrases)} phrases")

Total questions in dataset : 537,272
Sampling every 537th question  → 1001 phrases


In [188]:
# Compute phrase embeddings for all sampled questions
print("Computing phrase vectors...")
phrase_vectors = [get_phrase_embedding(phrase) for phrase in chosen_phrases]
print(f"Done! Shape: {len(phrase_vectors)} phrases × {len(phrase_vectors[0])} dimensions")

Computing phrase vectors...
Done! Shape: 1001 phrases × 100 dimensions


In [189]:
# Reduce phrase vectors to 2D with t-SNE, then standardize
print("Running t-SNE on phrase vectors (may take a few minutes)...")

phrase_vectors_2d = TSNE(n_components=2, init='random', learning_rate=200.0, verbose=2, random_state=42).fit_transform(
    np.array(phrase_vectors)
)

# Standardize: (x - mean) / std
phrase_vectors_2d = (
    (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0))
    / phrase_vectors_2d.std(axis=0)
)

print(f"\nt-SNE complete! Output shape: {phrase_vectors_2d.shape}")

Running t-SNE on phrase vectors (may take a few minutes)...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 1001 samples in 0.000s...
[t-SNE] Computed neighbors for 1001 samples in 0.015s...
[t-SNE] Computed conditional probabilities for sample 1000 / 1001
[t-SNE] Computed conditional probabilities for sample 1001 / 1001
[t-SNE] Mean sigma: 0.510829
[t-SNE] Computed conditional probabilities in 0.012s
[t-SNE] Iteration 50: error = 82.6912613, gradient norm = 0.1763169 (50 iterations in 0.137s)
[t-SNE] Iteration 100: error = 84.5624313, gradient norm = 0.2658312 (50 iterations in 0.142s)
[t-SNE] Iteration 150: error = 84.4916229, gradient norm = 0.2132046 (50 iterations in 0.149s)
[t-SNE] Iteration 200: error = 86.4121780, gradient norm = 0.2047274 (50 iterations in 0.139s)
[t-SNE] Iteration 250: error = 83.7998505, gradient norm = 0.6489949 (50 iterations in 0.131s)
[t-SNE] KL divergence after 250 iterations with early exaggeration: 83.799850
[t-SNE] Iteration 300: error = 1.

In [190]:
# Plot phrase embeddings — hover to see the first 50 characters of each question
draw_vectors(
    phrase_vectors_2d[:, 0],
    phrase_vectors_2d[:, 1],
    phrase=[p[:50] for p in chosen_phrases],  # show first 50 chars on hover
    radius=15,
    color='blue'
)

# Hover over clusters — can you find groups of similar questions?
# E.g., programming questions, political questions, relationship advice, etc.

figure(id='p1368', ...)

---
## Part 7 — Semantic Search: Find Similar Questions

### 📖 How it works

1. Convert the query to a phrase vector using `get_phrase_embedding`
2. Compute **cosine similarity** between the query vector and **all** question vectors
3. Return the top-k most similar questions

### 📖 Cosine Similarity vs. Cosine Distance

| Library | Function | Returns | Range |
|---------|----------|---------|-------|
| `scipy.spatial.distance` | `cosine(u, v)` | **Distance** | [0, 2] (0 = identical) |
| `sklearn.metrics.pairwise` | `cosine_similarity(U, V)` | **Similarity** | [−1, 1] (1 = identical) |

We use `sklearn`'s `cosine_similarity()`, which returns the **similarity** directly. (With scipy's `cosine()` you would first convert: **similarity = 1 − distance**.)

This ensures that `np.argsort(scores)[::-1]` (descending sort) correctly returns the **most similar** questions first.

### Step 7.1 — Pre-compute All Phrase Vectors

In [191]:
# Pre-compute phrase vectors for ALL questions in the dataset
# This takes a few minutes but needs to be done only once
print(f"Computing vectors for all {len(data):,} questions...")
print("(This may take 3–5 minutes)")

phrases_all = data  # keep reference to original text
phrases_all_vectors = np.array([get_phrase_embedding(phrase) for phrase in phrases_all])

print(f"\nDone! phrases_all_vectors shape: {phrases_all_vectors.shape}")

Computing vectors for all 537,272 questions...
(This may take 3–5 minutes)

Done! phrases_all_vectors shape: (537272, 100)


### Step 7.2 — The `find_nearest` Function

**Implementation details:**

- `cosine_similarity(phrases_all_vectors, query_vector.reshape(1, -1))` → **cosine similarity** (1=same, −1=opposite) between the query and every question, in one call; the result has shape [n × 1]
- `.ravel()` → flattens it into one score per question
- `np.argsort(similarities)[::-1]` → sort indices from highest to lowest similarity
- `[:k]` → keep only top-k indices

In [192]:
from sklearn.metrics.pairwise import cosine_similarity

def find_nearest(query, k=10):
    """
    Find the k most semantically similar questions to a query.

    Algorithm:
        1. Embed the query using get_phrase_embedding()
        2. Compute cosine similarity between query and every question vector
        3. Sort by similarity (highest first)
        4. Return the top-k questions

    Parameters:
        query (str): The input question or phrase to search for
        k (int)    : Number of results to return (default: 10)

    Returns:
        list[str]: The k most similar questions from the dataset
    """
    # Step 1: Embed the query
    query_vector = get_phrase_embedding(query)


    # Step 2: Compute cosine similarity for all questions (one vectorized call)
    cosine_dis = cosine_similarity(
        phrases_all_vectors,
        query_vector.reshape(1, -1)
    ).ravel()

    # Step 3: Sort by similarity descending (highest similarity first)
    sorted_index = np.argsort(cosine_dis)[::-1]

    # Step 4: Return the top-k questions
    topk_phrases = []
    for i in range(k):
        topk_phrases.append(phrases_all[sorted_index[i]])

    return topk_phrases


print("find_nearest() function defined.")

find_nearest() function defined.


### Step 7.3 — Run Semantic Search

In [193]:
# Search for questions similar to: "How do I enter the matrix?"
query = "How do i enter the matrix?"
results = find_nearest(query=query, k=10)

print(f'Query: "{query}"')
print(f"\nTop 10 most similar questions:")
print("-" * 60)
for i, r in enumerate(results, 1):
    print(f"  {i:2}. {r.strip()}")

Query: "How do i enter the matrix?"

Top 10 most similar questions:
------------------------------------------------------------
   1. How do I do a matrix transpose in Go?
   2. Do you live in the matrix? Why?
   3. If you could learn things like Neo in The Matrix, what is the one skill you would learn?
   4. How do I find the equation to this quartic function? (I'm doing precalc)
   5. How do I can get .in domain?
   6. How do you define the one?
   7. How do you define "the one"?
   8. How do we know that we don't live in a matrix?
   9. How do you know if you are in the friendzone?
  10. What do you do when you can't find a solution to an algorithms problem? And what do you do when you read the solution?


In [194]:
# Try another query
query2 = "How does Trump?"
results2 = find_nearest(query=query2, k=10)

print(f'Query: "{query2}"')
print(f"\nTop 10 most similar questions:")
print("-" * 60)
for i, r in enumerate(results2, 1):
    print(f"  {i:2}. {r.strip()}")

Query: "How does Trump?"

Top 10 most similar questions:
------------------------------------------------------------
   1. What does Tiffany Trump do?
   2. How would you describe Donald Trump?
   3. How does it feel to date Ivanka Trump?
   4. How would you characterize Melania Trump's personality?
   5. What is Trump?
   6. What companies does Donald Trump own?
   7. What promises can Donald Trump keep?
   8. What do you think Melania Trump really thinks of Donald?
   9. How does Donald Trump persuade?
  10. Why is Trump bad?


In [195]:
# Try your own query!
query3 = "Why don't i ask a question myself?"
results3 = find_nearest(query=query3, k=10)

print(f'Query: "{query3}"')
print(f"\nTop 10 most similar questions:")
print("-" * 60)
for i, r in enumerate(results3, 1):
    print(f"  {i:2}. {r.strip()}")

Query: "Why don't i ask a question myself?"

Top 10 most similar questions:
------------------------------------------------------------
   1. Why I don't get any answer on my question?
   2. Why I don't believe in myself?
   3. Why do you always answer a question with a question? I don't, or do I?
   4. Today my friend asked me how I know I'm bisexual if I've never dated a guy. How do I tell him that I don't need to?
   5. Why don't I feel like talking to anyone?
   6. How do I ask a girl I don't know to fuck?
   7. Why does it bother me that guys don't ask me out?
   8. Why don't I get a job call?
   9. What should I do if a girl loves me a lot but I don't?
  10. How can I ask someone I don't know to mentor me?


### Task 6 — A failure case for averaged phrase vectors

**Task:** Complete `phrase_similarity(a, b)`: the cosine similarity of `get_phrase_embedding(a)` and `get_phrase_embedding(b)`. The test compares the query *"How can I lose weight?"* with three candidates and also compares *"Is a dog smarter than a cat?"* with *"Is a cat smarter than a dog?"*. Then answer below:

- **(a)** Rank the three candidates. Why does the true paraphrase come last?
- **(b)** Why is the second similarity exactly 1?
- **(c)** Name one change to `get_phrase_embedding` that would help, and one problem it would still not fix.

**Why these candidates?** Each isolates one factor: same meaning with different words ("What is the best way to reduce body fat?"), the same "How can I …?" template on a different topic ("How can I learn Python?"), and the same template with the opposite goal ("How can I gain weight?"). "How can I" is chosen because templates like it dominate Quora, as the results of the matrix query above show.

In [196]:
QUERY = "How can I lose weight?"
CANDIDATES = ["What is the best way to reduce body fat?",    # same meaning, different words
              "How can I learn Python?",                     # same template, different topic
              "How can I gain weight?"]                      # same template, opposite goal


def phrase_similarity(a, b):
    """Return the cosine similarity between the phrase embeddings of a and b (a single number)."""
    return cosine_similarity(get_phrase_embedding(a).reshape(1, -1), get_phrase_embedding(b).reshape(1, -1))[0][0]

In [197]:
# ✅ Test — Task 6
try:
    task6_sims = {candidate: phrase_similarity(QUERY, candidate) for candidate in CANDIDATES}
    task6_swap = phrase_similarity("Is a dog smarter than a cat?", "Is a cat smarter than a dog?")
except NotImplementedError:
    print("⚠️  Task 6 is not finished yet. Complete phrase_similarity, re-run its cell, then re-run this test.")
else:
    print(f'Query: "{QUERY}"')
    for candidate, sim in sorted(task6_sims.items(), key=lambda item: -item[1]):
        print(f"  {sim:.4f}  {candidate}")
    print(f"\nDog/cat question vs the same words swapped: {task6_swap:.4f}")
    assert np.isclose(task6_sims[CANDIDATES[0]], 0.8925, atol=1e-3), f"Similarity to the paraphrase should be about 0.8925, got {task6_sims[CANDIDATES[0]]:.4f}. Use get_phrase_embedding for both phrases."
    assert np.isclose(task6_sims[CANDIDATES[1]], 0.9067, atol=1e-3), f"Similarity to the Python question should be about 0.9067, got {task6_sims[CANDIDATES[1]]:.4f}."
    assert np.isclose(task6_sims[CANDIDATES[2]], 0.9850, atol=1e-3), f"Similarity to the gain-weight question should be about 0.9850, got {task6_sims[CANDIDATES[2]]:.4f}."
    assert np.isclose(task6_swap, 1.0, atol=1e-5), f"Swapping the words should give a similarity of 1, got {task6_swap:.6f}."
    print("✅ Task 6 passed")

Query: "How can I lose weight?"
  0.9850  How can I gain weight?
  0.9067  How can I learn Python?
  0.8925  What is the best way to reduce body fat?

Dog/cat question vs the same words swapped: 1.0000
✅ Task 6 passed


# Answer

a. Ranking: "How can I gain weight?" (0.985), "How can I learn Python?" (0.907), "What is the best way to reduce body fat?" (0.893). The paraphrase is last because averaging is dominated by the common words "how can I" that the other two share. The paraphrase shares none of them.

b. Averaging ignores word order. Both questions have the same words, so they get the same vector and the similarity is exactly 1.

c. Give common words less weight (for example TF-IDF) so "how can I" stops dominating. It still would not fix word order, or opposites like "lose weight" vs "gain weight".

---
## Part 8 — Further Tasks

Tasks 7–10 follow the same format as Tasks 1–6: complete the function, run its test, and answer in the cell below the test. Each one builds on an earlier part of this lab: Task 7 on analogies and vector arithmetic (Part 4), Task 8 on Task 6, Task 9 on Task 7, and Task 10 on Word2Vec (Part 2).

**Submission**

- Work **in pairs** and write both names and student IDs in the table under the title.
- Complete **all ten tasks (Tasks 1–10)**.
- Support every written answer with the output of your code. Answers that do not match your output will not get full credit.
- Restart the kernel and **Run All** before you submit, so that every output is visible and comes from the final code.
- Submit **one notebook per pair**, named `MSAI6320_Embedding_<ID1>_<ID2>.ipynb`.

### Task 7 — One vector, two meanings

GloVe stores **one** vector per word, so a word with several meanings gets a single vector for all of them. Which meaning do its neighbours show, and is the other meaning still in the vector?

**Task:** Complete `remove_sense(word, anchors)`. First build a **sense direction** $\hat d$: average the unit vectors of the `anchors`, then scale the average to length 1. Then remove that direction from the word's vector, $\vec v-(\vec v\cdot\hat d)\,\hat d$, and return the result. The test prints the 10 nearest neighbours of *apple* and *java* before and after removing the sense described by their anchors, and runs the search engine from Part 7 for *"Is apple healthy?"* with and without the company sense of *apple*. Then answer below:

- **(a)** What do the neighbours of *apple* show before and after? Where was the fruit meaning before the removal, and why does the company meaning dominate the vector?
- **(b)** Why does *java* hardly change? Look at its neighbours before the removal.
- **(c)** Compare the two search results for *"Is apple healthy?"*. What does this show about searching with static word vectors, and which kind of model gives a word a different vector in each sentence?

**Why these anchors?** For *apple* they are four companies from its own top-10, so they describe the meaning that dominates its vector. For *java* they describe the programming language, the meaning most of you would expect. **Why unit vectors?** So that each anchor counts equally: a longer vector would pull the direction toward itself. **Why remove a single direction?** It deletes the anchor meaning while changing the vector as little as possible, because everything orthogonal to $\hat d$ is kept. **Why this query?** Its only content words are *apple* and *healthy*, so the sense of *apple* decides what the question is about, and to a reader *healthy* makes the fruit meaning obvious.

**Hint:** `model[w] / np.linalg.norm(model[w])` is the unit vector of `w`.

In [198]:
SENSES = {"apple": ["microsoft", "google", "intel", "ibm"],                  # the company
          "java":  ["software", "programming", "javascript", "computer"]}   # the programming language


def remove_sense(word, anchors):
    # Turn each anchor word into a unit vector (length 1) so every anchor counts equally
    units = [model[a] / np.linalg.norm(model[a]) for a in anchors]
    # Average the unit vectors: the direction the anchors share (the sense we want to remove)
    d = np.mean(units, axis=0)
    # The average is shorter than 1, so rescale it to length 1 to get the sense direction d
    d = d / np.linalg.norm(d)
    # Get the word's original vector
    v = model[word]
    # np.dot(v, d) is how much of v points along d; subtract that part to remove the sense
    return v - np.dot(v, d) * d

In [199]:
# ✅ Test — Task 7
def neighbours(vector, word, topn=10):
    """The topn words closest to `vector`, leaving out `word` itself."""
    return [w for w, _ in model.similar_by_vector(vector, topn=topn + 1) if w != word][:topn]


try:
    task7_vectors = {word: remove_sense(word, anchors) for word, anchors in SENSES.items()}
except NotImplementedError:
    print("⚠️  Task 7 is not finished yet. Complete remove_sense, re-run its cell, then re-run this test.")
else:
    task7_lists = {}
    for word, anchors in SENSES.items():
        assert np.shape(task7_vectors[word]) == (model.vector_size,), "remove_sense must return one vector of length 100."
        direction = np.mean([model[a] / np.linalg.norm(model[a]) for a in anchors], axis=0)
        leftover = np.dot(task7_vectors[word], direction) / (np.linalg.norm(task7_vectors[word]) * np.linalg.norm(direction))
        assert abs(leftover) < 1e-4, \
            f"The result for '{word}' still points along the sense direction (cosine {leftover:.3f}). Average the UNIT vectors of the anchors."
        task7_lists[word] = (neighbours(model[word], word), neighbours(task7_vectors[word], word))
        print(f"{word}  (removing: {', '.join(anchors)})")
        print(f"  before: {', '.join(task7_lists[word][0])}")
        print(f"  after : {', '.join(task7_lists[word][1])}\n")
    assert not set(SENSES["apple"]) & set(task7_lists["apple"][1]), "After the removal no company anchor should remain among apple's neighbours."
    assert "fruit" in task7_lists["apple"][1], "After the removal, 'fruit' should be among apple's 10 nearest neighbours."

    # The Part 7 search engine, once with the usual query vector and once with apple's company sense removed
    search_query = "Is apple healthy?"
    query_words = [w for w in tokenizer.tokenize(re.sub(r"[^a-zA-Z ]", " ", search_query).lower()) if w in model]
    query_vectors = {"usual vector for apple": get_phrase_embedding(search_query),
                     "company sense removed": np.mean([task7_vectors["apple"] if w == "apple" else model[w] for w in query_words], axis=0)}
    for name, query_vector in query_vectors.items():
        scores = cosine_similarity(phrases_all_vectors, query_vector.reshape(1, -1)).ravel()
        print(f"Top 6 for '{search_query}' ({name}):")
        for i in np.argsort(scores)[::-1][:6]:
            print("   ", phrases_all[i].strip())
        print()
    print("✅ Task 7 passed")

apple  (removing: microsoft, google, intel, ibm)
  before: microsoft, ibm, intel, software, dell, pc, macintosh, iphone, ipod, hewlett
  after : mango, peach, fruit, coconut, pear, citrus, apples, honey, cherry, pineapple

java  (removing: software, programming, javascript, computer)
  before: sumatra, surabaya, semarang, sulawesi, yogyakarta, bandung, mindanao, banten, borneo, luzon
  after : sumatra, sulawesi, surabaya, yogyakarta, semarang, banten, kalimantan, bandung, aceh, luzon

Top 6 for 'Is apple healthy?' (usual vector for apple):
    Is Apple dying?
    What is Apple?
    How healthy is the apple and tuna diet?
    How is Apple Music beneficial to Apple (2016)?
    How innovative is Apple?
    How is Apple innovative?

Top 6 for 'Is apple healthy?' (company sense removed):
    Is Chicken healthy?
    Is cheese healthy?
    How healthy is chocolate?
    How healthy is cereal?
    Is eating 3 eggs a day healthy?
    Is red wine healthy?

✅ Task 7 passed


# Answer

a. Before, the neighbours of *apple* are company words (microsoft, ibm, intel). After, they are fruit words (mango, peach, fruit). The fruit meaning was still in the vector but hidden. The company meaning dominates because *apple* appears mostly in company text.

b. *java* barely changes because its neighbours are already Indonesian islands (sumatra, surabaya, sulawesi), not programming, so there is little to remove.

c. The usual vector returns company questions ("Is Apple dying?"). With the company sense removed it returns food questions ("Is cheese healthy?"). Static vectors give a word one meaning everywhere. Contextual models like BERT give a different vector in each sentence.

### Task 8 — Repairing averaged phrase vectors

In Task 6 the plain average was dominated by the "How can I" template. Arora, Liang & Ma (2017) proposed a simple fix called **SIF** (smooth inverse frequency), in two steps:

1. **Weight** each word by $\dfrac{a}{a+p(w)}$, where $p(w)$ is the word's share of all tokens in the corpus. Frequent words get a small weight, and rare words a weight close to 1.
2. **Remove the common component:** find the direction $\hat u$ that all question vectors share (the first singular vector of the matrix of weighted question vectors) and subtract each vector's projection onto it, as in Task 7.

**Task:** Complete `sif_embedding(phrase)` (step 1: clean and tokenize exactly as `get_phrase_embedding` does, then average $\frac{a}{a+p(w)}\,\vec w$ over the words found in `model`) and `remove_common(vectors, direction)` (step 2, for a matrix with one vector per row). The test embeds all questions with your function, finds $\hat u$, and compares the three methods on Task 6's query and candidates, on the swapped dog/cat question, and on the "matrix" search from Part 7. Then answer below:

- **(a)** How does the ranking of the three candidates change from plain averaging to weighting, and then to removing the common component? Use the weights printed by the test.
- **(b)** Which failure from Task 6 survives both steps, and why can no choice of weights fix it?
- **(c)** Compare the top results for "How do i enter the matrix?". Which kind of irrelevant result disappears, and why?

**Why $a=10^{-3}$?** It is the value Arora et al. recommend. For a word with $p(w)\gg a$ the weight is about $a/p(w)$, so the most frequent words are pushed down strongly, while words rarer than about 1 in 1,000 tokens keep most of their weight. **Why $p(w)$ from Quora and not from GloVe's training text?** The weight should reflect how common a word is in the text we search: "how" is far more common in Quora questions than in Wikipedia. **Why `TruncatedSVD` rather than `PCA` for $\hat u$?** PCA first subtracts the mean vector, but the mean is exactly what all questions share. `TruncatedSVD` keeps it.

**Note:** The test embeds all 537,272 questions again, which takes about a minute.

**Hint:** Copy `get_phrase_embedding` and change one line. Words that never occur in Quora are not in `WORD_SHARE`; give them $p(w)=0$, i.e. weight 1.

In [200]:
from collections import Counter
from sklearn.decomposition import TruncatedSVD

SIF_A = 1e-3
# p(w): each word's share of all tokens in the Quora questions, after the same cleaning as get_phrase_embedding
quora_word_counts = Counter(word for question in data for word in tokenizer.tokenize(re.sub(r"[^a-zA-Z ]", " ", question).lower()))
quora_total = sum(quora_word_counts.values())
WORD_SHARE = {word: count / quora_total for word, count in quora_word_counts.items()}


def sif_embedding(phrase):
    """SIF step 1: the average of a/(a + p(w)) * model[w] over the words of `phrase` found in model (zero vector if none)."""
    # YOUR CODE HERE
    # Same cleaning and tokenizing as get_phrase_embedding (otherwise we would loop over characters)
    tokens = tokenizer.tokenize(re.sub(r"[^a-zA-Z ]", " ", phrase).lower())
    weights = [SIF_A / (SIF_A + WORD_SHARE.get(w, 0)) for w in tokens if w in model]
    vectors = [model[w] for w in tokens if w in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.average(vectors, axis=0, weights=weights)


def remove_common(vectors, direction):
    """SIF step 2: subtract from each row of `vectors` ([n × d]) its projection onto the unit vector `direction` ([d])."""
    # YOUR CODE HERE
    for i in range(vectors.shape[0]):
        vectors[i] -= np.dot(vectors[i], direction) * direction
    return vectors



In [201]:
# ✅ Test — Task 8
def task8_cosine(a, b):
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])


try:
    print("Embedding all questions with SIF weights (about a minute)...")
    task8_vectors = np.array([sif_embedding(question) for question in phrases_all])
    task8_direction = TruncatedSVD(n_components=1, n_iter=7, random_state=0).fit(task8_vectors).components_[0]
    task8_vectors_common = remove_common(task8_vectors, task8_direction)
except NotImplementedError:
    print("⚠️  Task 8 is not finished yet. Complete sif_embedding and remove_common, re-run their cell, then re-run this test.")
else:
    print("\nWeights a/(a + p(w)):", {w: round(SIF_A / (SIF_A + WORD_SHARE[w]), 3) for w in ["how", "can", "i", "python", "lose", "weight"]})
    task8_methods = {"plain":    get_phrase_embedding,
                     "weighted": sif_embedding,
                     "w − common": lambda p: remove_common(sif_embedding(p).reshape(1, -1), task8_direction)[0]}
    task8_rows = [(f'"{QUERY}" vs "{c}"', QUERY, c) for c in CANDIDATES] + \
                 [("dog/cat question vs words swapped", "Is a dog smarter than a cat?", "Is a cat smarter than a dog?")]
    task8_sims = {}
    print(f"\n  {'':<78}" + "".join(f"{name:>12}" for name in task8_methods))
    for label, a, b in task8_rows:
        task8_sims[label] = {name: task8_cosine(f(a), f(b)) for name, f in task8_methods.items()}
        print(f"  {label:<78}" + "".join(f"{task8_sims[label][name]:>12.3f}" for name in task8_methods))

    matrix_query = "How do i enter the matrix?"
    for name, matrix in [("plain", phrases_all_vectors), ("w − common", task8_vectors_common)]:
        scores = cosine_similarity(matrix, task8_methods[name](matrix_query).reshape(1, -1)).ravel()
        print(f"\nTop 6 for '{matrix_query}' ({name}):")
        for i in np.argsort(scores)[::-1][:6]:
            print("   ", phrases_all[i].strip())

    paraphrase, python_q, gain = (task8_sims[f'"{QUERY}" vs "{c}"'] for c in CANDIDATES)
    assert np.isclose(paraphrase["plain"], 0.8925, atol=1e-3), "The plain column should repeat Task 6's numbers."
    assert np.isclose(paraphrase["weighted"], 0.774, atol=0.02) and np.isclose(python_q["weighted"], 0.323, atol=0.02), \
        "Weighted similarities should be about 0.774 (paraphrase) and 0.323 (Python). Check the weight a/(a + p(w)) and the cleaning."
    assert paraphrase["w − common"] > python_q["w − common"], "After removing the common component, the paraphrase should rank above the Python question."
    assert abs(np.mean(task8_vectors_common @ task8_direction)) < 1e-4, "remove_common should leave no component along the direction."
    print("\n✅ Task 8 passed")

Embedding all questions with SIF weights (about a minute)...

Weights a/(a + p(w)): {'how': 0.043, 'can': 0.077, 'i': 0.039, 'python': 0.853, 'lose': 0.768, 'weight': 0.735}

                                                                                       plain    weighted  w − common
  "How can I lose weight?" vs "What is the best way to reduce body fat?"               0.892       0.774       0.459
  "How can I lose weight?" vs "How can I learn Python?"                                0.907       0.323      -0.101
  "How can I lose weight?" vs "How can I gain weight?"                                 0.985       0.901       0.785
  dog/cat question vs words swapped                                                    1.000       1.000       1.000

Top 6 for 'How do i enter the matrix?' (plain):
    How do I do a matrix transpose in Go?
    Do you live in the matrix? Why?
    If you could learn things like Neo in The Matrix, what is the one skill you would learn?
    How do I find th

# Answer

a. Plain averaging puts the paraphrase last (0.892). With weighting it moves above the Python question (0.774 vs 0.323), because "how", "can" and "I" count much less (about 0.04–0.08). Removing the common part keeps that order and spreads the scores further apart.

b. Word order is still not fixed: the dog/cat swap is still 1.000, and "gain weight" is still closest to "lose weight" (0.785) even though it means the opposite. Weights only change how much each word counts, and an average ignores order.

c. The results that only share "How do I ..." disappear. The top results are about *matrix*, because common words count less and the shared part is removed, so the rare word *matrix* decides.

### Task 9 — Does "debiasing" remove bias?

Word vectors learn the associations in their training text, including social stereotypes. Bolukbasi et al. (2016) proposed a fix: find a **gender direction** and **neutralize** words by removing their component along it, the same operation as in Task 7. Gonen & Goldberg (2019) then asked whether the bias is really gone.

**Task:** Complete two functions:

1. `gender_direction(pairs)`: for each (male, female) pair, take the difference of the two **unit** vectors and scale it to length 1; average these differences and scale the average to length 1.
2. `neutralize(vectors, direction)`: remove each row's component along `direction`.

The test projects 30 occupations onto the direction. It then takes the 500 most female-leaning and the 500 most male-leaning words among the 50,000 most frequent words, neutralizes them, and checks whether a vote among each word's 10 nearest neighbours (within these 1,000) still recovers the group it came from. Then answer below:

- **(a)** Which occupations lean most toward each side? Now look at the sample of the 1,000 words: apart from gender, what kinds of words fill each group, and what does that tell you about the text GloVe was trained on (Wikipedia and news)?
- **(b)** After neutralizing, every projection is zero, yet the neighbour vote still recovers the group of most words. How is that possible?
- **(c)** A recruiting tool matches CVs to job ads with these neutralized vectors. Is it free of gender bias? Use (b) in your answer.

**Why 8 pairs?** Each pair also carries meaning that has nothing to do with gender (*father − mother* also encodes parenthood); averaging pairs whose extra meanings differ cancels them out. **Why 500 + 500 words and 10 neighbours?** This follows the setup of Gonen & Goldberg (2019). **Why only alphabetic words?** Numbers and punctuation have no gender.

**Hint:** `neutralize` is the projection from Task 7, applied to every row: `vectors - np.outer(vectors @ direction, direction)`.

In [202]:
GENDER_PAIRS = [("he", "she"), ("man", "woman"), ("his", "her"), ("himself", "herself"),
                ("male", "female"), ("boy", "girl"), ("father", "mother"), ("son", "daughter")]
OCCUPATIONS = ["nurse", "receptionist", "homemaker", "librarian", "hairdresser", "nanny", "secretary", "dancer",
               "stylist", "maid", "engineer", "carpenter", "plumber", "mechanic", "architect", "surgeon", "programmer",
               "pilot", "firefighter", "captain", "doctor", "lawyer", "teacher", "scientist", "manager", "professor",
               "chef", "writer", "banker", "farmer"]


def gender_direction(pairs):
    """Return the unit gender direction: the normalised average of the normalised differences unit(male) − unit(female)."""
    # YOUR CODE HERE
    for male, female in pairs:
        male_vector = model[male] / np.linalg.norm(model[male])
        female_vector = model[female] / np.linalg.norm(model[female])
        diff = male_vector - female_vector
        diff /= np.linalg.norm(diff)
        if 'avg' in locals():
            avg += diff
        else:
            avg = diff
    avg /= len(pairs)
    avg /= np.linalg.norm(avg)
    return avg  


def neutralize(vectors, direction):
    """Remove from each row of `vectors` ([n × d]) its component along the unit vector `direction` ([d])."""
    # Return a new array (np.outer builds the projection of every row), so the input is not overwritten
    return vectors - np.outer(vectors @ direction, direction)


In [203]:
# ✅ Test — Task 9
def neighbour_vote_accuracy(vectors, labels, k=10):
    """Share of rows whose k nearest other rows (cosine) have the same majority label."""
    unit_rows = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    similarity = unit_rows @ unit_rows.T
    np.fill_diagonal(similarity, -np.inf)                       # a word may not vote for itself
    nearest = np.argsort(-similarity, axis=1)[:, :k]
    return float(np.mean((labels[nearest].mean(axis=1) > 0.5) == labels))


try:
    task9_direction = gender_direction(GENDER_PAIRS)
except NotImplementedError:
    print("⚠️  Task 9 is not finished yet. Complete gender_direction and neutralize, re-run their cell, then re-run this test.")
else:
    assert np.isclose(np.linalg.norm(task9_direction), 1.0, atol=1e-5), "The gender direction must have length 1."
    unit_vector = lambda w: model[w] / np.linalg.norm(model[w])
    assert unit_vector("he") @ task9_direction > 0, "The direction should point from female to male (male minus female)."
    task9_projections = sorted(((float(unit_vector(w) @ task9_direction), w) for w in OCCUPATIONS))
    print("Most female-leaning:", ", ".join(f"{w} ({p:+.2f})" for p, w in task9_projections[:5]))
    print("Most male-leaning  :", ", ".join(f"{w} ({p:+.2f})" for p, w in task9_projections[::-1][:5]))

    frequent = [w for w in sorted(model.key_to_index, key=lambda w: model.get_vecattr(w, "count"), reverse=True)[:50_000] if w.isalpha()]
    frequent_vectors = np.array([unit_vector(w) for w in frequent])
    order = np.argsort(frequent_vectors @ task9_direction)
    chosen = np.r_[order[:500], order[-500:]]                    # 500 most female-leaning, then 500 most male-leaning
    task9_labels = np.r_[np.zeros(500), np.ones(500)]            # 0 = female group, 1 = male group
    print("\nSample, female group:", ", ".join(frequent[i] for i in order[:12]))
    print("Sample, male group  :", ", ".join(frequent[i] for i in order[::-1][:12]))

    before = frequent_vectors[chosen]
    after = neutralize(before, task9_direction)
    task9_max_projection = float(np.abs(after @ task9_direction).max())
    task9_accuracy = (neighbour_vote_accuracy(before, task9_labels), neighbour_vote_accuracy(after, task9_labels))
    print(f"\nLargest |projection| after neutralizing: {task9_max_projection:.1e}")
    print(f"Neighbour vote recovers the group: before {task9_accuracy[0]:.3f}, after {task9_accuracy[1]:.3f}   (chance: 0.5)")
    assert task9_max_projection < 1e-5, "After neutralize, no word should have a component along the direction."
    assert task9_accuracy[1] > 0.85, f"Expected the vote to stay well above chance after neutralizing, got {task9_accuracy[1]:.3f}."
    print("✅ Task 9 passed")

Most female-leaning: homemaker (-0.36), maid (-0.34), receptionist (-0.31), nurse (-0.30), nanny (-0.29)
Most male-leaning  : manager (+0.31), captain (+0.27), architect (+0.21), engineer (+0.20), mechanic (+0.18)

Sample, female group: irina, rhoda, kristina, louisa, kournikova, gina, uterus, pregnant, caroline, femininity, michelle, congresswoman
Sample, male group  : caliph, arsene, yawar, gundy, harun, vogts, chiefs, parreira, umpires, caliphate, defensive, umayyad

Largest |projection| after neutralizing: 8.9e-08
Neighbour vote recovers the group: before 1.000, after 0.925   (chance: 0.5)
✅ Task 9 passed


# Answer

a. Female: homemaker, maid, receptionist, nurse, nanny. Male: manager, captain, architect, engineer, mechanic. The female group is mostly first names and female words (irina, pregnant), and the male group is sports and history/religion words (umpires, caliph). So Wikipedia and news write about men and women in different topics, and GloVe copies that.

b. Neutralizing removes only one direction. The words in each group still share topics (names, sports), so they stay close and the vote still works (1.000 before, 0.925 after, chance is 0.5).

c. No. The gender pattern is still in the other directions, so the tool can still match CVs and jobs using gender-linked words.

### Task 10 — Counting instead of predicting: raw counts vs PPMI

Before Word2Vec, word vectors were built by **counting** how often each word appears near each other word. The code cell below builds this word–context count matrix $C$ from `data_tok` for the 10,000 most frequent Quora tokens with a ±2 window. The test reduces its rows to 32 dimensions with truncated SVD, the same size as our Word2Vec vectors in Part 2.

Raw counts are dominated by frequent words: almost every word appears near "?", "the" and "i". **Positive pointwise mutual information** asks instead whether two words appear together *more often than chance*:

$$\text{PPMI}(w,c)=\max\!\left(0,\ \log\frac{P(w,c)}{P(w)\,P(c)}\right),\qquad P(w,c)=\frac{C_{wc}}{N},\quad P(w)=\frac{\sum_{c}C_{wc}}{N},\quad P(c)=\frac{\sum_{w}C_{wc}}{N},\quad N=\sum_{w,c}C_{wc}$$

PMI compares the probability of seeing $w$ and $c$ together with the probability we would expect if the two words were independent, $P(w)\,P(c)$. A ratio of 3 means "three times more often than chance", and its log, the PMI, is then positive. **A worked example:** the code cell below prints PMI for (*what*, *the*), the most frequent pair in $C$, and for (*acoustic*, *guitar*), a pair that is rare but meaningful. Compare their counts with their PMI before you start. **Why clip at 0?** A negative PMI says two words appear together *less* often than chance. Estimating that reliably needs far more text than we have, because most word pairs are rare, so PPMI replaces negative values with 0.

**Task:** Complete `ppmi(counts)` for a sparse count matrix. Compute the value only for the non-zero counts (a zero count would give $\log 0$, which PPMI sets to 0 anyway) and return a sparse matrix of the same shape. The test checks your function on a 2 × 2 example, builds 32-D vectors from the raw counts and from PPMI, prints the neighbours of three words, and scores both with a **GloVe agreement score**: for 20 probe words, the average GloVe similarity between the word and its 10 nearest Quora neighbours. Then answer below:

- **(a)** Compare the neighbours and the agreement scores. Where do raw counts and PPMI sit between the random floor and the GloVe ceiling printed by the test?
- **(b)** Why do raw counts give poor neighbours? Use the largest counts in $C$, printed by the test.
- **(c)** Levy & Goldberg (2014) showed that Skip-gram with negative sampling implicitly factorizes a PMI matrix shifted by $\log k$, where $k$ is the number of negative samples. What is $k$ for our Part 2 model? What does this result say about "counting" versus "predicting" methods?

**Why GloVe as a judge?** It was trained on 6 billion tokens of other text, so its similarities are an independent reference. The floor (random words as neighbours) and the ceiling (GloVe's own neighbours) show what "bad" and "perfect" look like on this scale. **Why the 10,000 most frequent tokens?** Each occurs at least about 30 times, enough to estimate its counts, and a 10,000 × 10,000 matrix is small when stored sparsely. **Why a ±2 window?** A small window favours words that can replace each other (Task 1), which is what a list of nearest neighbours should show.

**Note:** Your numbers may differ slightly from your classmates' if your tokenizer version differs (see the note in Part 2).

**Hint:** `counts.tocoo()` gives the arrays `.row`, `.col` and `.data` of the non-zero entries, and `np.asarray(counts.sum(axis=1)).ravel()` the row sums. Build the result with `sp.csr_matrix((values, (rows, cols)), shape=counts.shape)`.

In [204]:
import scipy.sparse as sp
from collections import Counter
from sklearn.decomposition import TruncatedSVD

COUNT_VOCAB_SIZE, COUNT_WINDOW, COUNT_DIM = 10_000, 2, 32

token_counts = Counter(token for question in data_tok for token in question)
count_vocab = [word for word, _ in token_counts.most_common(COUNT_VOCAB_SIZE)]
count_index = {word: i for i, word in enumerate(count_vocab)}

# One long array of word ids. -1 marks words outside the vocabulary and the gaps between questions,
# so that no window reaches from one question into the next.
count_ids = []
for question in data_tok:
    count_ids.extend(count_index.get(token, -1) for token in question)
    count_ids.extend([-1] * COUNT_WINDOW)
count_ids = np.array(count_ids)

pair_rows, pair_cols = [], []
for distance in range(1, COUNT_WINDOW + 1):          # each (word, word `distance` positions later) pair, counted in both directions
    left, right = count_ids[:-distance], count_ids[distance:]
    keep = (left >= 0) & (right >= 0)
    pair_rows += [left[keep], right[keep]]
    pair_cols += [right[keep], left[keep]]
pair_rows, pair_cols = np.concatenate(pair_rows), np.concatenate(pair_cols)
counts = sp.coo_matrix((np.ones(len(pair_rows)), (pair_rows, pair_cols)),
                       shape=(COUNT_VOCAB_SIZE, COUNT_VOCAB_SIZE)).tocsr()    # duplicate pairs are summed
print(f"Count matrix: {counts.shape[0]:,} × {counts.shape[1]:,}, {counts.nnz:,} non-zero entries "
      f"({counts.nnz / counts.shape[0] ** 2:.1%}), {int(counts.sum()):,} word–context pairs")
print(f"Least frequent token in the vocabulary occurs {token_counts[count_vocab[-1]]} times")

# A worked example of PMI for two pairs: the most frequent pair in C, and a rare one
total_pairs = counts.sum()
print("\nWorked example:")
for word, context in [("what", "the"), ("acoustic", "guitar")]:
    i, j = count_index[word], count_index[context]
    p_wc = counts[i, j] / total_pairs
    p_w, p_c = counts[i].sum() / total_pairs, counts[:, j].sum() / total_pairs
    print(f"  ({word}, {context}): count {int(counts[i, j]):>6,}   P(w,c) = {p_wc:.2e}   P(w)·P(c) = {p_w * p_c:.2e}   "
          f"ratio = {p_wc / (p_w * p_c):6.1f}   PMI = log(ratio) = {np.log(p_wc / (p_w * p_c)):5.2f}")


def ppmi(counts):
    """Return the PPMI matrix of a sparse count matrix, as a sparse matrix of the same shape."""
    # Only the non-zero entries: coo.row, coo.col, coo.data (a zero count would give log 0)
    coo = counts.tocoo()
    N = coo.data.sum()
    # One total per word (rows) and per context (columns)
    row_sums = np.asarray(counts.sum(axis=1)).ravel()
    col_sums = np.asarray(counts.sum(axis=0)).ravel()
    # PMI = log(P(w,c) / (P(w) P(c))) for each non-zero entry
    pmi = np.log((coo.data / N) / ((row_sums[coo.row] / N) * (col_sums[coo.col] / N)))
    # The "positive" in PPMI: replace negative values with 0
    pmi = np.maximum(pmi, 0)
    return sp.csr_matrix((pmi, (coo.row, coo.col)), shape=counts.shape)


Count matrix: 10,000 × 10,000, 2,553,138 non-zero entries (2.6%), 23,091,336 word–context pairs
Least frequent token in the vocabulary occurs 30 times

Worked example:
  (what, the): count 82,509   P(w,c) = 3.57e-03   P(w)·P(c) = 8.57e-04   ratio =    4.2   PMI = log(ratio) =  1.43
  (acoustic, guitar): count     54   P(w,c) = 2.34e-06   P(w)·P(c) = 8.67e-10   ratio = 2696.3   PMI = log(ratio) =  7.90


In [205]:
# ✅ Test — Task 10
PROBES = ["python", "doctor", "guitar", "india", "money", "exam", "girlfriend", "engineering", "pizza", "car",
          "university", "weight", "job", "movie", "phone", "bitcoin", "football", "religion", "election", "cancer"]


def svd_vectors(matrix):
    return TruncatedSVD(n_components=COUNT_DIM, n_iter=10, random_state=0).fit_transform(matrix)


def count_neighbours(vectors, word, topn=10):
    unit_rows = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    order = np.argsort(-(unit_rows @ unit_rows[count_index[word]]))
    return [count_vocab[i] for i in order if count_vocab[i] != word][:topn]


def glove_agreement(neighbour_lists):
    return float(np.mean([np.mean([model.similarity(w, n) for n in ns if n in model]) for w, ns in neighbour_lists.items()]))


try:
    tiny = ppmi(sp.csr_matrix(np.array([[0.0, 4.0], [4.0, 2.0]])))
except NotImplementedError:
    print("⚠️  Task 10 is not finished yet. Complete ppmi, re-run its cell, then re-run this test.")
else:
    assert sp.issparse(tiny), "ppmi must return a sparse matrix."
    expected = np.log(0.4 / (0.4 * 0.6))
    assert np.allclose(tiny.toarray(), [[0.0, expected], [expected, 0.0]], atol=1e-6), \
        f"On [[0, 4], [4, 2]] PPMI should be [[0, {expected:.3f}], [{expected:.3f}, 0]], got {tiny.toarray().round(3).tolist()}."

    coo = counts.tocoo()
    print("Largest counts in C:", ", ".join(f"({count_vocab[coo.row[i]]}, {count_vocab[coo.col[i]]}) {int(coo.data[i]):,}" for i in np.argsort(-coo.data)[:6]))

    print("Building 32-D vectors from raw counts and from PPMI...")
    task10_raw, task10_ppmi = svd_vectors(counts), svd_vectors(ppmi(counts))
    for word in ["doctor", "money", "guitar"]:
        print(f"\n{word}\n  raw : {', '.join(count_neighbours(task10_raw, word, 5))}\n  PPMI: {', '.join(count_neighbours(task10_ppmi, word, 5))}")

    rng = np.random.default_rng(0)
    task10_scores = {
        "random floor":  glove_agreement({w: [x for x in rng.choice(count_vocab, 10) if x != w] for w in PROBES}),
        "raw counts":    glove_agreement({w: count_neighbours(task10_raw, w) for w in PROBES}),
        "PPMI":          glove_agreement({w: count_neighbours(task10_ppmi, w) for w in PROBES}),
        "GloVe ceiling": glove_agreement({w: [n for n, _ in model.most_similar(w, topn=10)] for w in PROBES}),
    }
    print("\nGloVe agreement (mean GloVe similarity of each probe word to its 10 neighbours):")
    for name, score in task10_scores.items():
        print(f"  {name:<14}{score:.3f}")
    assert task10_scores["PPMI"] > task10_scores["raw counts"] + 0.1, "PPMI should agree with GloVe clearly better than raw counts."
    print("✅ Task 10 passed")

Largest counts in C: (what, the) 82,509, (the, what) 82,509, (what, is) 79,856, (is, what) 79,856, (the, is) 69,331, (is, the) 69,331
Building 32-D vectors from raw counts and from PPMI...

doctor
  raw : man, soldier, pilot, person, gun
  PPMI: therapist, teacher, child, partner, doctors

money
  raw : weight, come, sleep, patience, practice
  PPMI: cash, pay, funds, bitcoin, stocks

guitar
  raw : store, app, wii, violin, rti
  PPMI: acoustic, violin, piano, bass, jazz

GloVe agreement (mean GloVe similarity of each probe word to its 10 neighbours):
  random floor  0.171
  raw counts    0.358
  PPMI          0.538
  GloVe ceiling 0.726
✅ Task 10 passed


# Answer

a. PPMI neighbours make sense (guitar: acoustic, violin, piano), raw counts look random (guitar: store, app, wii). Scores: random 0.171, raw counts 0.358, PPMI 0.538, GloVe 0.726. Raw counts are about a third of the way up, PPMI about two thirds.

b. The biggest counts are very common words: (what, the) 82,509, (what, is) 79,856, (the, is) 69,331. Almost every word appears near them, so raw counts show how common a word is, not what it means.

c. k = 10 (our Part 2 model uses negative=10). Skip-gram secretly does the same thing as a PMI count matrix, so counting and predicting are closely related.

---
## 🎯 Summary & Key Takeaways

| Concept | What you learned |
|---------|------------------|
| **Word2Vec** | Trains word vectors by predicting context. Requires a corpus and training time. |
| **GloVe** | Pre-trained on 6B tokens (Wikipedia + Gigaword). Ready to use, much richer. |
| **Cosine Similarity** | Measures semantic closeness between vectors (range: −1 to 1). |
| **Analogies** | Vector arithmetic encodes semantic relationships: `king − man + woman ≈ queen`. |
| **PCA** | Fast linear reduction. Good for global structure. |
| **t-SNE** | Slow non-linear reduction. Excellent for revealing local clusters. |
| **Phrase Embeddings** | Average word vectors to represent phrases/sentences. |
| **Semantic Search** | Rank questions by cosine similarity to a query vector. |

---

## 🚀 Going Further (Optional Explorations)

1. **Larger t-SNE plot:** Run t-SNE on all ~400k phrases (will take longer but shows more structure).

2. **Explore the Gensim model zoo:**
   ```python
   import gensim.downloader as api
   print(list(api.info()['models'].keys()))
   ```

3. **FastText embeddings:** Unlike GloVe/Word2Vec, FastText handles **out-of-vocabulary** words by decomposing them into character n-grams. Try loading `fasttext-wiki-news-subwords-300`.

4. **Speed up `find_nearest`:** For large datasets, brute-force cosine similarity is slow. Try locality-sensitive hashing with [`nearpy`](https://github.com/pixelogik/NearPy) or `sklearn.neighbors.NearestNeighbors`.

5. **Bias in embeddings:** Investigate gender bias with:  
   ```python
   analogy("man", "doctor", "woman")      # What does this return?
   analogy("woman", "doctor", "man")      # And this?
   ```

---
*End of Lab*

---
## 🚀 AI Assisted Optional Explorations (one cell per item above)

Quick attempts at the five optional explorations. Item 1 uses 5,000 phrases instead of all ~400k to keep it fast; the FastText cell downloads a large file (~1 GB).

In [206]:
# Optional 1: larger t-SNE plot (5,000 phrases instead of 1,000)
big_step = len(data) // 5000
big_phrases = data[::big_step]
big_vectors = np.array([get_phrase_embedding(p) for p in big_phrases])

big_2d = TSNE(n_components=2, init='random', learning_rate=200.0, random_state=42).fit_transform(big_vectors)
big_2d = (big_2d - big_2d.mean(axis=0)) / big_2d.std(axis=0)

draw_vectors(big_2d[:, 0], big_2d[:, 1], phrase=[p[:50] for p in big_phrases], radius=8, color='blue')

figure(id='p1419', ...)

In [207]:
# Optional 2: explore the Gensim model zoo
import gensim.downloader as api

print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [208]:
# Optional 3: FastText (large download, ~1 GB)
# Note: this gensim-data file may only store whole-word vectors, so an unseen word can still fail.
ft = api.load("fasttext-wiki-news-subwords-300")

for word in ["pizza", "pizzaaaaa"]:            # a normal word and a misspelled (likely out-of-vocabulary) one
    try:
        print(word, "->", ft.most_similar(word, topn=3))
    except KeyError:
        print(word, "-> not in the FastText vocabulary either")

pizza -> [('pizzas', 0.8317782282829285), ('pizza.', 0.7641924023628235), ('Pizza', 0.7259705662727356)]
pizzaaaaa -> not in the FastText vocabulary either


In [209]:
# Optional 4: faster find_nearest
# Normalise every row once; then cosine similarity is just a dot product, and
# np.argpartition finds the top-k without sorting all 537,272 scores.
import time

row_norms = np.linalg.norm(phrases_all_vectors, axis=1, keepdims=True)
phrases_all_unit = phrases_all_vectors / np.maximum(row_norms, 1e-12)      # avoid dividing by 0 for empty phrases


def find_nearest_fast(query, k=10):
    q = get_phrase_embedding(query)
    q = q / max(np.linalg.norm(q), 1e-12)
    scores = phrases_all_unit @ q
    top = np.argpartition(-scores, k)[:k]            # the k best, in no particular order
    top = top[np.argsort(-scores[top])]              # sort just those k
    return [phrases_all[i] for i in top]


for fn in (find_nearest, find_nearest_fast):
    start = time.time()
    out = fn("How do i enter the matrix?", k=5)
    print(f"{fn.__name__:<18}{time.time() - start:.3f} s   {out[0].strip()}")

find_nearest      0.227 s   How do I do a matrix transpose in Go?
find_nearest_fast 0.005 s   How do I do a matrix transpose in Go?


In [210]:
# Optional 5: bias in embeddings
print("man : doctor :: woman :", analogy("man", "doctor", "woman"))
print("woman : doctor :: man :", analogy("woman", "doctor", "man"))

man : doctor :: woman : nurse
woman : doctor :: man : dr.
